In [1]:
import pandas as pd
import sklearn

df = pd.read_csv("../data/processed/BRinf_clean.csv", index_col=0, parse_dates=True)

df.head(5)

,ipca_mom,igpm_mom,igpdi_mom,fgv_igp_unknown,igp10_mom,focus_ipca_12m_median,electricity_consumption_total,electricity_consumption_ind,bofa_economic_condition_idx,cni_capacity_util_diff,...,focus_top5_med_t12,focus_top5_med_t13,focus_med_t1_sq,focus_mean_t1,focus_mean_t1_sq,focus_std_t1,focus_med_t2_sq,focus_mean_t2,focus_mean_t2_sq,focus_std_t2
2003-03-31,1.23,1.53,1.66,4.593,1.58,9.58,24960.0,115.0,-0.35,-0.3,...,0.55,0.50,0.8836,0.93,0.8649,0.13,0.4900,0.73,0.5329,0.17
2003-04-30,0.97,0.92,0.41,1.173,1.24,8.88,25284.0,93.0,-0.15,-0.5,...,0.45,0.78,1.0000,0.97,0.9409,0.14,0.2500,0.52,0.2704,0.18
2003-05-31,0.61,-0.26,-0.67,-1.860,0.02,8.22,24785.0,285.0,-0.47,0.0,...,0.60,0.43,0.2500,0.50,0.2500,0.10,0.3025,0.55,0.3025,0.15
2003-06-30,-0.15,-1.00,-0.70,-1.949,-0.59,6.91,24889.0,-99.0,-0.73,0.1,...,0.51,0.55,0.0100,0.17,0.0289,0.17,1.1025,1.07,1.1449,0.21
2003-07-31,0.20,-0.42,-0.20,-0.544,-0.73,6.29,24536.0,38.0,-0.14,-0.5,...,0.50,0.97,0.0900,0.39,0.1521,0.28,0.3364,0.56,0.3136,0.18


In [2]:
df.shape

(154, 92)

In [3]:
df['ipca_next_month'] = df['ipca_mom'].shift(-1)

df.head(5)

,ipca_mom,igpm_mom,igpdi_mom,fgv_igp_unknown,igp10_mom,focus_ipca_12m_median,electricity_consumption_total,electricity_consumption_ind,bofa_economic_condition_idx,cni_capacity_util_diff,...,focus_top5_med_t13,focus_med_t1_sq,focus_mean_t1,focus_mean_t1_sq,focus_std_t1,focus_med_t2_sq,focus_mean_t2,focus_mean_t2_sq,focus_std_t2,ipca_next_month
2003-03-31,1.23,1.53,1.66,4.593,1.58,9.58,24960.0,115.0,-0.35,-0.3,...,0.50,0.8836,0.93,0.8649,0.13,0.4900,0.73,0.5329,0.17,0.97
2003-04-30,0.97,0.92,0.41,1.173,1.24,8.88,25284.0,93.0,-0.15,-0.5,...,0.78,1.0000,0.97,0.9409,0.14,0.2500,0.52,0.2704,0.18,0.61
2003-05-31,0.61,-0.26,-0.67,-1.860,0.02,8.22,24785.0,285.0,-0.47,0.0,...,0.43,0.2500,0.50,0.2500,0.10,0.3025,0.55,0.3025,0.15,-0.15
2003-06-30,-0.15,-1.00,-0.70,-1.949,-0.59,6.91,24889.0,-99.0,-0.73,0.1,...,0.55,0.0100,0.17,0.0289,0.17,1.1025,1.07,1.1449,0.21,0.20
2003-07-31,0.20,-0.42,-0.20,-0.544,-0.73,6.29,24536.0,38.0,-0.14,-0.5,...,0.97,0.0900,0.39,0.1521,0.28,0.3364,0.56,0.3136,0.18,0.34


In [4]:
df.tail(1)

df = df.drop("2015-12-31")

In [5]:
df.shape


(153, 93)

## train and test split

In [6]:
df_train = df.loc[:"2011-12"]
df_test = df.loc["2012-01":]

df_train.head(3)

,ipca_mom,igpm_mom,igpdi_mom,fgv_igp_unknown,igp10_mom,focus_ipca_12m_median,electricity_consumption_total,electricity_consumption_ind,bofa_economic_condition_idx,cni_capacity_util_diff,...,focus_top5_med_t13,focus_med_t1_sq,focus_mean_t1,focus_mean_t1_sq,focus_std_t1,focus_med_t2_sq,focus_mean_t2,focus_mean_t2_sq,focus_std_t2,ipca_next_month
2003-03-31,1.23,1.53,1.66,4.593,1.58,9.58,24960.0,115.0,-0.35,-0.3,...,0.50,0.8836,0.93,0.8649,0.13,0.4900,0.73,0.5329,0.17,0.97
2003-04-30,0.97,0.92,0.41,1.173,1.24,8.88,25284.0,93.0,-0.15,-0.5,...,0.78,1.0000,0.97,0.9409,0.14,0.2500,0.52,0.2704,0.18,0.61
2003-05-31,0.61,-0.26,-0.67,-1.860,0.02,8.22,24785.0,285.0,-0.47,0.0,...,0.43,0.2500,0.50,0.2500,0.10,0.3025,0.55,0.3025,0.15,-0.15


In [7]:
X_train = df_train.drop(columns="ipca_next_month")
y_train = df_train["ipca_next_month"]

X_test = df_test.drop(columns="ipca_next_month")
y_test = df_test["ipca_next_month"]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(106, 92)
(106,)
(47, 92)
(47,)


In [8]:
erro = (y_test - X_test["ipca_mom"]).abs()

mae = erro.mean()
rmse = ((erro ** 2).mean()) ** 0.5

print(f"mae: {mae}\nrmse: {rmse}")

mae: 0.20212765957446807
rmse: 0.24323178683646507


In [9]:
focus_t1 = (y_test - X_test["focus_med_t1"]).abs()
focus_t2 = (y_test - X_test["focus_med_t2"]).abs()

mae_focus_t1 = focus_t1.mean()
rmse_focus_t1 = ((focus_t1**2).mean())**0.5

mae_focus_t2 = focus_t2.mean()
rmse_focus_t2 = ((focus_t2**2).mean())**0.5

print(f"mae_t1 = {mae_focus_t1}\nrmse_t1 = {rmse_focus_t1}\nmae_t2 = {mae_focus_t2}\nrmse_t2 = {rmse_focus_t2}")

mae_t1 = 0.19425531914893615
rmse_t1 = 0.2316132138956321
mae_t2 = 0.15276595744680851
rmse_t2 = 0.18507042937700025


## First training LASSO


In [14]:
scaler = sklearn.preprocessing.StandardScaler()

X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

In [15]:
lasso = sklearn.linear_model.Lasso(alpha=0.1)

lasso.fit(X_train_normalized, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [16]:
lasso_predictions = lasso.predict(X_test_normalized)

mae_lasso = sklearn.metrics.mean_absolute_error(y_test, lasso_predictions)
rmse_lasso = sklearn.metrics.root_mean_squared_error(y_test, lasso_predictions)

print(f"mae_lasso = {mae_lasso}\nrmse_lasso = {rmse_lasso}")

mae_lasso = 0.2081237392110362
rmse_lasso = 0.2691770298885547


In [22]:
count = 0
for c in lasso.coef_:
    if c == 0:
        count += 1

print(count)


90


In [28]:
columns = df.columns.drop("ipca_next_month")

for c, v in pd.Series(lasso.coef_, index=columns).items():
    if v != 0:
        print(c, v)


ipca_mom 0.02438579355086214
focus_top5_med_t2 0.030592003282063657


In [44]:
split = sklearn.model_selection.TimeSeriesSplit(n_splits=5)

lassoCV = sklearn.linear_model.LassoCV(cv=split, max_iter=10000)

lassoCV.fit(X_train_normalized, y_train)

c:\Projetos\Previsao-IPCA\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.674886e-04, tolerance: 3.963e-04
  model = cd_fast.enet_coordinate_descent(
c:\Projetos\Previsao-IPCA\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.278828e-03, tolerance: 3.963e-04
  model = cd_fast.enet_coordinate_descent(
c:\Projetos\Previsao-IPCA\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duali

,"max_iter max_iter: int, default=1000The maximum number of iterations.",10000
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"alphas alphas: array-like or int, default=100Values of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`.",100
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as an argument.",'auto'
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.0001
,"copy_X copy_X: bool, default=TrueIf ``True``, `X` will be copied; otherwise, it may be overwritten.",True
,"verbose verbose: bool or int, default=FalseAmount of verbosity.",False
,"n_jobs n_jobs: int, default=NoneNumber of CPUs to use during the cross validation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"positive positive: bool, default=FalseIf positive, restrict regression coefficients to be positive.",False


In [45]:
lassoCV.alpha_

np.float64(0.014172307843385414)

In [46]:
lassoCV_predictions = lassoCV.predict(X_test_normalized)

mae_lassoCV = sklearn.metrics.mean_absolute_error(y_test, lassoCV_predictions)
rmse_lassoCV = sklearn.metrics.root_mean_squared_error(y_test, lassoCV_predictions)

print(f"mae_lassoCV = {mae_lassoCV}\nrmse_lassoCV = {rmse_lassoCV}")

mae_lassoCV = 0.15093493540672837
rmse_lassoCV = 0.18983970645794554


In [48]:
for c,v in pd.Series(lassoCV.coef_, index=columns).items():
    if v != 0:
        print(c,v)

ipca_mom 0.053917587238727346
igpdi_mom 0.038440898700398775
industry_working_hours -0.022836832845639067
ibovespa 0.02652504953016534
savings_deposits_rate -0.005275298928728377
tjlp 0.014815094323157307
pis_pasep_tax -0.004061824563578231
importing_tax_income 0.0012952335784637537
trade_balance_fob 0.0026446762475731537
public_net_gov_bcb_debt 0.003745270165757869
m3 0.03521966735823382
m4 0.0040200149999295096
focus_top5_med_t2 0.0908378437534822
focus_top5_med_t7 -0.029609510422001953
focus_top5_med_t13 0.010698421840808


In [42]:
random_forest = sklearn.ensemble.RandomForestRegressor(random_state=42)

random_forest.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [43]:
rf_predictions = random_forest.predict(X_test)

mae_random_forest = sklearn.metrics.mean_absolute_error(y_test, rf_predictions)
rmse_random_forest = sklearn.metrics.root_mean_squared_error(y_test, rf_predictions)

print(f"mae_random_forest = {mae_random_forest}\nrmse_random_forest = {rmse_random_forest}")

mae_random_forest = 0.16690000000000005
rmse_random_forest = 0.2187870814771216


In [49]:
results = pd.DataFrame([
    {"modelo": "repeat_last_value", "mae": mae.round(3), "rmse": rmse.round(3)},
    {"modelo": "focus", "mae": mae_focus_t2.round(3), "rmse": rmse_focus_t2.round(3)},
    {"modelo": "lasso(alpha=0.1)", "mae": round(mae_lasso,3), "rmse": round(rmse_lasso,3)},
    {"modelo": "lassoCV(alpha=0.0141)", "mae": round(mae_lassoCV,3), "rmse": round(rmse_lassoCV,3)},
    {"modelo": "random_forest", "mae": round(mae_random_forest,3), "rmse": round(rmse_random_forest,3)}
])

results

,modelo,mae,rmse
0,repeat_last_value,0.202,0.243
1,focus,0.153,0.185
2,lasso(alpha=0.1),0.208,0.269
3,lassoCV(alpha=0.0141),0.151,0.190
4,random_forest,0.167,0.219
